In [1]:
import pandas as pd
import numpy as np
from scipy.stats import qmc

ModuleNotFoundError: No module named 'scipy'

In [ ]:
lista_zavorra = [0, 20, 40, 60]
campioni_per_zavorra = 50

limiti_inferiori = [0.0, -4.5, 0.0, 1.5]
limiti_superiori = [6.0, -2.0, 0.3, 4.5]

risultati = []

#LHS
sampler = qmc.LatinHypercube(d=4, seed=42)

for zavorra in lista_zavorra:
    campioni_lhs = sampler.random(n=campioni_per_zavorra)
    campioni_scalati = qmc.scale(campioni_lhs, limiti_inferiori, limiti_superiori)

    for riga in campioni_scalati:
        vento = round(riga[0], 2)
        coeff_teta = round(riga[1], 2)
        std_vento = round(riga[2], 3)
        delay = round(riga[3], 2)

        teta_rampa = round(vento * coeff_teta, 2)

        risultati.append({
            'vento_medio': vento,
            'std_vento': round(std_vento*vento,2),
            'zavorra': zavorra,
            'delay_secondi': delay,
            'angolo_rampa': teta_rampa,
            'APOGEO_MISURATO': '',
            'ANGOLO_ACCENSIONE_C6': ''
        })

df = pd.DataFrame(risultati)
df.to_csv("Simulazioni_GEMINI.csv", index=False)

In [ ]:
from sklearn.preprocessing import StandardScaler as SS

df = pd.read_csv("Simulazioni_GEMINI.csv")

#CLEANING
for column in df.columns:
  df[column] = df[column].astype("str")
  df[column] = df[column].str.replace(",", ".")
  df[column] = df[column].astype("float")

#SHUFFLE
df = df.sample(frac=1, random_state=999).reset_index(drop=True)

#LABELS
y1 = df["APOGEO_MISURATO"]
y2 = df["ANGOLO_ACCENSIONE_C6"]

#PREDICTIVE VARIABLES
x = df.drop(["APOGEO_MISURATO", "ANGOLO_ACCENSIONE_C6"], axis=1)

#NORMALIZATION
scaler = SS()
x = scaler.fit_transform(x)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
import xgboost as xgb

#RANDOM FOREST
modello_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=999,
    n_jobs=-1)

#XGBOOST
modello_xgb = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=999,
    n_jobs=-1)

#SUPPORT VECTOR REGRESSION
modello_svr = SVR(
    kernel='rbf',
    C=50 #100,
    gamma='scale')